# 09 money-based threshold

## Cost assumptions and result

For each model and fee, the threshold is selected using validation labels, probabilities, and `Amount_raw` only. Every alert costs the fee; an unflagged fraud costs its raw transaction amount. The test set is evaluated once with the selected threshold. Savings are measured against the test split’s no-review fraud loss of 12,973.06 amount units. The fees 5, 10, and 25 are assumptions in the same units as `Amount_raw`, and the calculation assumes a reviewed fraud is fully prevented.

At fee 10, the F1 threshold saved 9,287.36 units for XGBoost and 9,626.34 for fastai on test. Validation-cost thresholds saved 5,793.01 and 9,626.34 respectively. The test comparison is descriptive; it does not change threshold selection. See `results/money_thresholds.csv` for caught frauds, false alarms, costs, and savings for every model and fee.

## Student-authored explanation

Write the cost assumptions and results interpretation here in your own words.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO = Path.cwd()
if REPO.name == "notebooks":
    REPO = REPO.parent
if not (REPO / "src" / "data.py").exists():
    REPO = Path("/content/credit-card-fraud-detection")
    if not (REPO / "src" / "data.py").exists():
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/VanshSharmaPES/credit-card-fraud-detection.git",
             str(REPO)],
            check=True,
        )
os.chdir(REPO)
%pip install -q -r requirements/base.txt
!python src/data.py
sys.path.insert(0, str(REPO / "src"))


In [1]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

from cost import best_cost_threshold, cost_breakdown, threshold_cost_curve
from data import ROOT, load_splits

(X_train, y_train, X_val, y_val, X_test, y_test,
 row_id_val, row_id_test, amount_val, amount_test) = load_splits()

val_amounts = pd.DataFrame({
    "row_id": np.asarray(row_id_val),
    "y_true_data": np.asarray(y_val, dtype=int),
    "Amount_raw": np.asarray(amount_val, dtype=float),
})
test_amounts = pd.DataFrame({
    "row_id": np.asarray(row_id_test),
    "y_true_data": np.asarray(y_test, dtype=int),
    "Amount_raw": np.asarray(amount_test, dtype=float),
})

def predictions_with_amount(model_name, split):
    prediction_path = ROOT / "preds" / f"{model_name}_{split}.csv"
    if not prediction_path.exists():
        raise FileNotFoundError(prediction_path)
    predictions = pd.read_csv(prediction_path)
    amounts = val_amounts if split == "val" else test_amounts
    joined = predictions.merge(amounts, on="row_id", how="left", validate="one_to_one")
    if joined["Amount_raw"].isna().any():
        raise ValueError(f"Missing Amount_raw values for {model_name} {split} predictions")
    if not np.array_equal(joined["y_true"].to_numpy(), joined["y_true_data"].to_numpy()):
        raise ValueError(f"Label mismatch after row_id join for {model_name} {split}")
    return joined

model_names = sorted(
    path.name[:-8]
    for path in (ROOT / "preds").glob("*_val.csv")
    if (ROOT / "preds" / f"{path.name[:-8]}_test.csv").exists()
    and (ROOT / "results" / f"{path.name[:-8]}.json").exists()
)
if not model_names:
    raise RuntimeError("No model prediction pairs and result JSON files were found")

FEE_VALUES = (5.0, 10.0, 25.0)
money_records = []
val_predictions = {}
for model_name in model_names:
    val = predictions_with_amount(model_name, "val")
    test = predictions_with_amount(model_name, "test")
    val_predictions[model_name] = val
    result = json.loads((ROOT / "results" / f"{model_name}.json").read_text())

    for fee in FEE_VALUES:
        chosen = best_cost_threshold(
            val["y_true"], val["y_prob"], val["Amount_raw"], fee
        )
        val_cost = cost_breakdown(
            val["y_true"], val["y_prob"], val["Amount_raw"], chosen, fee
        )
        test_cost = cost_breakdown(
            test["y_true"], test["y_prob"], test["Amount_raw"], chosen, fee
        )
        money_records.append({
            "model": model_name,
            "decision_rule": "validation cost minimum",
            "validation_threshold": chosen,
            "validation_total_cost": val_cost["total_cost"],
            **test_cost,
        })

    fee = 10.0
    f1_threshold = float(result["threshold"])
    f1_test_cost = cost_breakdown(
        test["y_true"], test["y_prob"], test["Amount_raw"], f1_threshold, fee
    )
    money_records.append({
        "model": model_name,
        "decision_rule": "validation F1 maximum",
        "validation_threshold": f1_threshold,
        "validation_total_cost": cost_breakdown(
            val["y_true"], val["y_prob"], val["Amount_raw"], f1_threshold, fee
        )["total_cost"],
        **f1_test_cost,
    })

money_table = pd.DataFrame(money_records)
money_table.to_csv(ROOT / "results" / "money_thresholds.csv", index=False)
display_columns = [
    "model", "review_fee", "decision_rule", "threshold", "frauds_caught",
    "false_alarms", "review_cost", "total_cost", "money_saved",
]
display(money_table[display_columns].sort_values(["review_fee", "model", "decision_rule"]))

fee10 = money_table[money_table["review_fee"] == 10.0]
f1_vs_cost = fee10.pivot(
    index="model", columns="decision_rule", values="money_saved"
).rename(columns={
    "validation F1 maximum": "f1_threshold_money_saved",
    "validation cost minimum": "cost_threshold_money_saved",
})
f1_vs_cost["cost_minus_f1_savings"] = (
    f1_vs_cost["cost_threshold_money_saved"] - f1_vs_cost["f1_threshold_money_saved"]
)
f1_vs_cost["higher_test_savings"] = np.where(
    f1_vs_cost["cost_minus_f1_savings"] > 0, "cost threshold", "F1 threshold"
)
f1_vs_cost.to_csv(ROOT / "results" / "money_f1_vs_cost.csv")
display(f1_vs_cost.reset_index())

fee10 = money_table[money_table["review_fee"] == 10.0]
f1_vs_cost = fee10.pivot(
    index="model", columns="decision_rule", values="money_saved"
).rename(columns={
    "validation F1 maximum": "f1_threshold_money_saved",
    "validation cost minimum": "cost_threshold_money_saved",
})
f1_vs_cost["cost_minus_f1_savings"] = (
    f1_vs_cost["cost_threshold_money_saved"] - f1_vs_cost["f1_threshold_money_saved"]
)
f1_vs_cost["higher_test_savings"] = np.where(
    f1_vs_cost["cost_minus_f1_savings"] > 0, "cost threshold", "F1 threshold"
)
f1_vs_cost.to_csv(ROOT / "results" / "money_f1_vs_cost.csv")
display(f1_vs_cost.reset_index())

validation_rank = sorted(
    (
        model_name,
        average_precision_score(predictions["y_true"], predictions["y_prob"]),
    )
    for model_name, predictions in val_predictions.items()
)
best_model = max(validation_rank, key=lambda item: item[1])[0]
curve_val = val_predictions[best_model]
curve_thresholds, curve_costs = threshold_cost_curve(
    curve_val["y_true"], curve_val["y_prob"], curve_val["Amount_raw"], fee=10.0
)
curve_df = pd.DataFrame({"threshold": curve_thresholds, "validation_cost": curve_costs})
curve_df = curve_df.sort_values("threshold")
best_threshold_fee10 = best_cost_threshold(
    curve_val["y_true"], curve_val["y_prob"], curve_val["Amount_raw"], fee=10.0
)

(ROOT / "figures").mkdir(exist_ok=True)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(curve_df["threshold"], curve_df["validation_cost"], color="#147D83")
ax.axvline(best_threshold_fee10, color="#C35A3A", linestyle="--", label="Validation cost minimum")
ax.set_xlabel("Probability threshold")
ax.set_ylabel("Validation total cost (Amount_raw units)")
ax.set_title(f"Validation cost curve: {best_model}, review fee 10")
ax.legend()
fig.tight_layout()
fig.savefig(ROOT / "figures" / "money_cost_curve.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)

cost_rows = money_table[money_table["decision_rule"] == "validation cost minimum"]
savings = cost_rows.pivot(index="model", columns="review_fee", values="money_saved")
ax = savings.plot(kind="bar", figsize=(10, 5), color=["#147D83", "#5CA4A8", "#C35A3A"])
ax.set_xlabel("Model")
ax.set_ylabel("Test money saved (Amount_raw units)")
ax.set_title("Test savings after selecting each threshold on validation")
ax.legend(title="Review fee")
plt.tight_layout()
plt.savefig(ROOT / "figures" / "money_saved_all.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close()


model  review_fee           decision_rule  threshold  frauds_caught  false_alarms  review_cost  total_cost  money_saved
 ensemble         5.0 validation cost minimum   0.927702             75             4        395.0     3310.70      9662.36
   fastai         5.0 validation cost minimum   0.998328             76            10        430.0     2916.72     10056.34
    keras         5.0 validation cost minimum   0.966736             74            19        465.0     6425.90      6547.16
lightning         5.0 validation cost minimum   0.904274             83            78        805.0     2540.46     10432.60
   tabnet         5.0 validation cost minimum   0.999842             67            17        420.0     4239.55      8733.51
  xgboost         5.0 validation cost minimum   0.783177             76             4        400.0     3314.70      9658.36
 ensemble        10.0   validation F1 maximum   0.927702             75             4        790.0     3705.70      9267.36
 ensemble   